## Setup

### General

In [ ]:
import logging
import os
import sys

import numpy as np
import pandas as pd

from _utilities.stupidloggen import StupidLogGenerator


### Paths & Logger

In [ ]:
ROOT_DIR = os.environ["VIRTUAL_ENV"][:-5]
sys.path.append(ROOT_DIR)

PATH_ORG_DATA = "data/compas-scores-two-years.csv"
PATH_SYNTHETIC_DATA = "data/synthetic/"
PATH_FOLDER_MODELS = "models/"

if not os.path.exists(PATH_FOLDER_MODELS):
    os.makedirs(PATH_FOLDER_MODELS)
if not os.path.exists(PATH_SYNTHETIC_DATA):
    os.makedirs(PATH_SYNTHETIC_DATA)

In [ ]:
logger = StupidLogGenerator().initLogger()

### Dataset

Sensitive attributes include:
- sex
- race

Target variable:
- is_recid

In [ ]:
scores_two_years = pd.read_csv(PATH_ORG_DATA)

columns_to_include = [
    "sex",
    "age",
    "age_cat",
    "race",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
    "priors_count",
    "c_charge_degree",
    "c_charge_desc",
    "r_charge_degree",
    "r_charge_desc",
    "two_year_recid",
    "is_recid",
]

dataset = scores_two_years[columns_to_include].copy()

In [ ]:
categorical_columns = [
    "sex",
    "age_cat",
    "race",
    "c_charge_degree",
    "c_charge_desc",
    "r_charge_degree",
    "r_charge_desc",
    "two_year_recid",
    "is_recid",
]

In [ ]:
dataset.nunique()

## Data Preprocessing

When `is_recid` is 0 there should not be any values in `r_charge_degree` and `r_charge_desc`, as they contain information about the crime they were re-arested for. Thus, we made a separate category "Not applicable" to make it clear which values are truly missing.

In [ ]:
recid_charge_cols = [
    "r_charge_degree",
    "r_charge_desc",
]

for col in recid_charge_cols:
    dataset[col] = np.where((dataset["is_recid"] == 0), "Not applicable", dataset[col])

The `race` attribute is distributed as indicated in the table below. The ammount of data on "Hispanic", "Asian", and "Native American" individually is not enough to be significant, therefore we combine these together with "Other".

| Race             | Count | Percent |
| --- | --- | ---  |
| African-American |  3696 |  51.23% |
| Caucasian        |  2454 |  34.02% |
| Hispanic         |   637 |   8.83% |
| Other            |   377 |   5.23% |
| Asian            |    32 |   0.44% |
| Native American  |    18 |   0.25% |

In [ ]:
other_races = ["Hispanic", "Asian", "Native American"]

dataset["race"] = dataset["race"].replace(other_races, "Other")

In [ ]:
dataset.to_csv("data/preprocessed_real_data.csv", index=False)

## Model Training

#### Model Setup

In [ ]:
# To make sure that FairGAN can be imported
path_to_dir = os.environ["VIRTUAL_ENV"][:-5]
sys.path.append(path_to_dir)

In [ ]:
import pickle

from ctgan import CTGAN, TVAE
from tabfairgan import TFG

from fairgan import FairGAN


In [ ]:
def save_successful(model, model_name: str, synthetic_data: pd.DataFrame):
    try:
        synthetic_data.to_csv(
            PATH_SYNTHETIC_DATA + f"{model_name}_synthetic.csv", index=False
        )

        with open(PATH_FOLDER_MODELS + f"{model_name}_model.pkl", "wb") as f:
            pickle.dump(model, f)
    except Exception as e:
        logger.error(e)

#### CTGAN

The CTGAN model learns the joint distribution of the dataset.

In [ ]:
def run_CTGAN(
    df: pd.DataFrame,
    num_epochs: int,
    batch_size: int,
    verbose: bool,
    synthetic_data_count: int,
):
    logger.info("\n\nStarting to train CTGAN model")

    try:
        # Train CTGAN model
        ctgan_df = df.copy()

        ctgan_model = CTGAN(epochs=num_epochs, batch_size=batch_size, verbose=verbose)
        ctgan_model.fit(ctgan_df, discrete_columns=categorical_columns)

        # Generate synthetic data using CTGAN
        ctgan_synthetic_data = ctgan_model.sample(synthetic_data_count)

        save_successful(ctgan_model, "ctgan", ctgan_synthetic_data)

    except Exception as e:
        logger.error(e)

#### TVAE

TVAE learns a latent representation of the data and reconstructs it through a decoder network.

Unlike CTGAN, TVAE optimizes a probabilistic objective rather than using adversarial training.

In [ ]:
def run_TVAE(
    df: pd.DataFrame,
    num_epochs: int,
    batch_size: int,
    verbose: bool,
    synthetic_data_count: int,
):
    logger.info("\n\nStarting to train TVAE model")

    try:
        # Train TVAE model
        tvae_df = df.copy()

        tvae_model = TVAE(epochs=num_epochs, batch_size=batch_size, verbose=verbose)
        tvae_model.fit(tvae_df, discrete_columns=categorical_columns)

        # Generate synthetic data using TVAE
        tvae_synthetic_data = tvae_model.sample(synthetic_data_count)

        save_successful(tvae_model, "tvae", tvae_synthetic_data)
    except Exception as e:
        logger.error(e)

#### FairGAN

Our implementation of the FairGAN model can be found [here](fairgan.py)

In [ ]:
def run_FairGAN(
    df: pd.DataFrame,
    sensitive_feature: str,
    target_feature: str,
    pretrain_epochs: int,
    epochs: int,
    batch_size: int,
    noise_dim: int,
    lr: float,
    verbose: bool,
):
    logger.info("\n\nStarting to train FairGAN with different lambda_fair values")

    lambda_values = [0.0, 0.1, 0.5, 0.7, 1.0]

    fairgan_datasets = {}
    fairgan_models = {}

    for lam in lambda_values:
        logger.info(f"\nTraining FairGAN with lambda_fair = {lam}")

        fairgan_df = df.copy()

        try:
            # Initialize model
            fairgan_model = FairGAN(
                df=fairgan_df,
                sensitive_feature=sensitive_feature,
                target_feature=target_feature,
                pretrain_epochs=pretrain_epochs,
                epochs=epochs,
                batch_size=batch_size,
                noise_dim=noise_dim,
                lr=lr,
                verbose=verbose,
            )

            # Train model
            fairgan_model.train(lambda_fair=lam)

            # Generate synthetic dataset
            synthetic_df = fairgan_model.generate(synthetic_data_count)

            dataset_name = f"FairGAN_lambda_{lam}"
            fairgan_datasets[dataset_name] = synthetic_df
            fairgan_models[dataset_name] = fairgan_model

            save_successful(fairgan_model, dataset_name, synthetic_df)

            logger.info(f"Completed {dataset_name}")

        except Exception as e:
            logger.error(e)

#### TabFairGAN

TabFairGAN differs from standard generative models such as CTGAN in that fairness constraints are incorporated directly into the training process.

Instead of using a separate fit function, the model is trained using a custom training loop.

**Fairness Constraints**

To generate fair synthetic data, we need to pass a dictionary containing the following parameters:

- `fair_epochs`: Number of fair training epochs (integer).
- `lamda`: Lambda parameter controlling the trade-off between fairness and accuracy (float).
- `S`: Protected attribute (string, e.g., "sex").
- `Y`: Decision label (string, e.g., "income").
- `S_under`: Value representing the underprivileged group for the protected attribute (string, e.g., " Female").
- `Y_desire`: Desired value for the label (string, e.g., " >50K").

In [ ]:
# TODO: Define proper Y_desire


def run_TabFairGAN(
    df: pd.DataFrame,
    num_epochs: int,
    batch_size: int,
    synthetic_data_count: int,
    fairness_config: dict,
):
    logger.info("\n\nStarting to train TabFairGAN with different lambda values")

    lambda_values = [0.0, 0.1, 0.5, 0.7, 1.0]

    tabfairgan_datasets = {}
    tabfairgan_models = {}

    for lam in lambda_values:
        if lam == 0.0:
            fairness_config["fair_epochs"] = 0
        else:
            fairness_config["fair_epochs"] = 50

        fairness_config["lamda"] = lam

        try:
            logger.info(f"\nTraining TabFairGAN with lambda = {lam}")

            tfg_df = df.copy()

            # Turn the sensitive and target attribute into categorical string
            tfg_df["sex"] = tfg_df["sex"].astype(str)
            tfg_df["is_recid"] = tfg_df["is_recid"].astype(str)

            # Initialize model
            tfg_model = TFG(
                tfg_df,
                epochs=num_epochs,
                batch_size=batch_size,
                fairness_config=fairness_config,
            )

            # Train model
            tfg_model.train()

            # Generate synthetic data
            tfg_synthetic_data = tfg_model.generate_fake_df(
                num_rows=synthetic_data_count
            )

            dataset_name = f"TabFairGAN_lambda_{lam}"
            tabfairgan_datasets[dataset_name] = tfg_synthetic_data
            tabfairgan_models[dataset_name] = tfg_model

            save_successful(tfg_model, dataset_name, tfg_synthetic_data)

            logger.info(f"Completed {dataset_name}")

        except Exception as e:
            logger.error(e)
            continue

## Model Settings

In [ ]:
# Set logging level
logger.setLevel(logging.INFO)

In [ ]:
target_feature = "is_recid"
sensitive_feature = "sex"
sensitive_under = "Female"

fairness_config = {
    "S": sensitive_feature,
    "Y": target_feature,
    "S_under": sensitive_under,
    "Y_desire": "0",
}

In [ ]:
verbose = 2
synthetic_data_count = 5000
num_epochs = 200
batch_size = 120

# FairGAN hyperparameters
num_pretrain_epochs = 20
fairgan_noise_dim = 100
fairgan_lr = 0.001

## Run models

In [ ]:
run_CTGAN(dataset, num_epochs, batch_size, verbose, synthetic_data_count)
run_TVAE(dataset, num_epochs, batch_size, verbose, synthetic_data_count)

run_FairGAN(
    dataset,
    sensitive_feature,
    target_feature,
    pretrain_epochs=num_pretrain_epochs,
    epochs=num_epochs,
    batch_size=batch_size,
    noise_dim=fairgan_noise_dim,
    lr=fairgan_lr,
    verbose=verbose,
)

run_TabFairGAN(dataset, num_epochs, batch_size, synthetic_data_count, fairness_config)
